# Explore latest datasets snapshot

Tinkering notebook to decide which fields the slim historic files should keep
(see README "Open decisions"). Reads columns **remotely** via `HfFileSystem` +
parquet column projection — only the requested columns are transferred, not the
full 377 MB file.

## SetUp

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
from dotenv import load_dotenv
from huggingface_hub import HfFileSystem

load_dotenv()
fs = HfFileSystem()

LATEST = "2026-07-15"  # last week in the snapshot repo as of 2026-07-22
PATH = f"datasets/hfmlsoc/hub_weekly_snapshots/datasets/{LATEST}/datasets.parquet"

## Schema + per-column size (footer only, no data transfer)

In [ ]:
from collections import defaultdict

f = pq.ParquetFile(fs.open(PATH))
md = f.metadata
sizes = defaultdict(int)
for rg in range(md.num_row_groups):
    g = md.row_group(rg)
    for ci in range(g.num_columns):
        col = g.column(ci)
        sizes[col.path_in_schema.split(".")[0]] += col.total_compressed_size

print(f"rows: {md.num_rows:,} | total compressed: {sum(sizes.values())/1e6:.0f} MB")
for name in f.schema_arrow.names:
    print(f"{name:20s} {str(f.schema_arrow.field(name).type):25s} {sizes[name]/1e6:8.1f} MB")

## Load selected columns

Edit `COLS` to pull whatever you want to inspect. Avoid `cardData`/`description`/`sha`
unless needed — they are the heavy ones (204/56/39 MB).

In [ ]:
COLS = ["_id", "id", "author", "likes", "downloads", "downloadsAllTime",
        "trendingScore", "tags", "mainSize", "gated", "createdAt"]

df = f.read(columns=COLS).to_pandas()
df.head()

## Open question: paper references

`paperswithcode_id` is not the only paper carrier — `tags` contains `arxiv:XXXX.XXXXX`
entries and `citation` holds free-text BibTeX. Run this to compare coverage:

In [ ]:
t = f.read(columns=["tags", "paperswithcode_id", "citation"])
n = t.num_rows
arxiv = sum(1 for row in t.column("tags").to_pylist()
            if row and any(x.startswith("arxiv:") for x in row))
pwc = n - t.column("paperswithcode_id").null_count
cit = sum(1 for c in t.column("citation").to_pylist() if c)
print(f"rows {n:,}")
print(f"arxiv tag:          {arxiv:8,} ({arxiv/n:.1%})")
print(f"paperswithcode_id:  {pwc:8,} ({pwc/n:.1%})")
print(f"citation non-empty: {cit:8,} ({cit/n:.1%})")